In [2]:
import duckdb, numpy as np, pandas as pd, matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score

con = duckdb.connect("../data/processed/recalls.duckdb", read_only=True)
df = con.execute("SELECT * FROM panel").df()
con.close()

df["month"] = pd.to_datetime(df["month"])
for c in ["make", "system"]:
    df[c] = df[c].astype("category")

FEATURES = ["make", "system", "n_1m", "n_3m", "n_6m", "n_12m", "n_prev3m",
            "crash_12m", "fire_12m", "injury_12m", "towed_12m",
            "vehicle_age", "months_since_first", "vehicle_12m", "system_share",
            "sibling_years_12m", "family_recalled"]
CAT = ["make", "system"]
print(df.shape)

(2363007, 24)


In [3]:
train = df[df["month"] < "2022-01-01"]
test  = df[df["month"] >= "2023-01-01"]

X_tr, y_tr = train[FEATURES], train["target"]
X_te, y_te = test[FEATURES], test["target"]

for name, d in [("train", train), ("test", test)]:
    print(f"{name}: {len(d):,} rows, {d['target'].mean():.3%} positive, "
          f"{d.loc[d.target==1,'campaign'].nunique()} recalls")

train: 1,459,336 rows, 0.799% positive, 921 recalls
test: 717,696 rows, 1.031% positive, 648 recalls


In [4]:
def score(name, y, p):
    return {"model": name, "ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p)}

results = [
    score("Base rate", y_te, np.full(len(y_te), y_tr.mean())),
    score("Complaint volume (n_12m)", y_te, X_te["n_12m"].fillna(0)),
    score("Recent surge (n_3m)", y_te, X_te["n_3m"].fillna(0)),
    score("Sibling year recalled", y_te, X_te["family_recalled"].fillna(0)),
]
print(pd.DataFrame(results).round(4).to_string(index=False))

                   model  ROC-AUC  PR-AUC
               Base rate   0.5000  0.0103
Complaint volume (n_12m)   0.6127  0.0191
     Recent surge (n_3m)   0.6107  0.0196
   Sibling year recalled   0.5756  0.0124


In [5]:
model = lgb.LGBMClassifier(
    n_estimators=600, learning_rate=0.05, num_leaves=63, min_child_samples=200,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1)
model.fit(X_tr, y_tr, categorical_feature=CAT)
p = model.predict_proba(X_te)[:, 1]

results.append(score("LightGBM", y_te, p))
res = pd.DataFrame(results).round(4)
print(res.to_string(index=False))
res.to_csv("../results/model_comparison.csv", index=False)

                   model  ROC-AUC  PR-AUC
               Base rate   0.5000  0.0103
Complaint volume (n_12m)   0.6127  0.0191
     Recent surge (n_3m)   0.6107  0.0196
   Sibling year recalled   0.5756  0.0124
                LightGBM   0.8105  0.0383


In [6]:
te = test.copy()
te["p"] = p
te["rank"] = te.groupby("month")["p"].rank(ascending=False, method="first")
total_recalls = te.loc[te.target == 1, "campaign"].nunique()

rows = []
for k in [25, 50, 100, 250]:
    top = te[te["rank"] <= k]
    rows.append({"alerts_per_month": k,
                 "precision": top["target"].mean(),
                 "lift_vs_random": top["target"].mean() / te["target"].mean(),
                 "recalls_caught": top.loc[top.target == 1, "campaign"].nunique(),
                 "of_total": total_recalls})
print(pd.DataFrame(rows).round(3).to_string(index=False))

 alerts_per_month  precision  lift_vs_random  recalls_caught  of_total
               25      0.087           8.439              28       648
               50      0.082           7.954              42       648
              100      0.074           7.129              76       648
              250      0.065           6.286             143       648


In [7]:
imp = pd.Series(model.booster_.feature_importance(importance_type="gain"),
                index=FEATURES).sort_values(ascending=False)
print(imp.round(0).to_string())

vehicle_age           129891.0
make                  127334.0
system                101304.0
sibling_years_12m      99541.0
months_since_first     78770.0
system_share           63710.0
vehicle_12m            61289.0
n_12m                  25562.0
family_recalled        23431.0
n_6m                   23369.0
crash_12m              10603.0
towed_12m               8738.0
n_3m                    6645.0
injury_12m              5442.0
fire_12m                4392.0
n_prev3m                3482.0
n_1m                    1158.0


In [8]:
def fit_eval(feats, label):
    cats = [c for c in CAT if c in feats]
    m = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63,
                           min_child_samples=200, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1)
    m.fit(X_tr[feats], y_tr, categorical_feature=cats)
    return score(label, y_te, m.predict_proba(X_te[feats])[:, 1])

family = ["sibling_years_12m", "family_recalled"]
severity = ["crash_12m", "fire_12m", "injury_12m", "towed_12m"]

abl = pd.DataFrame([
    score("All features", y_te, p),
    fit_eval([f for f in FEATURES if f not in family], "Without family features"),
    fit_eval([f for f in FEATURES if f not in severity], "Without severity"),
    fit_eval([f for f in FEATURES if f in family + CAT], "Family features only"),
]).round(4)
print(abl.to_string(index=False))
abl.to_csv("../results/ablation.csv", index=False)

                  model  ROC-AUC  PR-AUC
           All features   0.8105  0.0383
Without family features   0.8079  0.0367
       Without severity   0.8080  0.0381
   Family features only   0.6634  0.0206


In [11]:
structural = ["make", "system", "vehicle_age"]
complaint_signal = ["n_1m", "n_3m", "n_6m", "n_12m", "n_prev3m",
                    "crash_12m", "fire_12m", "injury_12m", "towed_12m",
                    "vehicle_12m", "system_share", "sibling_years_12m", "months_since_first"]

abl2 = pd.DataFrame([
    fit_eval(structural, "Structural only (make, system, age)"),
    fit_eval(complaint_signal, "Complaint signals only"),
    fit_eval(structural + complaint_signal, "Structural + complaints (no family)"),
    score("All features", y_te, p),
]).round(4)
print(abl2.to_string(index=False))
abl2.to_csv("../results/ablation_structure.csv", index=False)

                              model  ROC-AUC  PR-AUC
Structural only (make, system, age)   0.7797  0.0323
             Complaint signals only   0.7045  0.0252
Structural + complaints (no family)   0.8091  0.0375
                       All features   0.8105  0.0383


In [12]:
m_s = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63,
                         min_child_samples=200, subsample=0.8, subsample_freq=1,
                         colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1)
m_s.fit(X_tr[structural], y_tr, categorical_feature=["make", "system"])
te["p_struct"] = m_s.predict_proba(X_te[structural])[:, 1]
te["rank_struct"] = te.groupby("month")["p_struct"].rank(ascending=False, method="first")

rows = []
for k in [25, 50, 100, 250]:
    full = te[te["rank"] <= k]
    st = te[te["rank_struct"] <= k]
    rows.append({"alerts": k,
                 "precision_full": full["target"].mean(),
                 "precision_struct": st["target"].mean(),
                 "recalls_full": full.loc[full.target == 1, "campaign"].nunique(),
                 "recalls_struct": st.loc[st.target == 1, "campaign"].nunique()})
print(pd.DataFrame(rows).round(3).to_string(index=False))

 alerts  precision_full  precision_struct  recalls_full  recalls_struct
     25           0.087             0.060            28              16
     50           0.082             0.070            42              35
    100           0.074             0.067            76              55
    250           0.065             0.052           143             107


In [9]:
import pickle
pickle.dump({"model": model, "features": FEATURES, "cat": CAT},
            open("../results/model.pkl", "wb"))
te[["make", "model", "model_year", "system", "month", "p", "target",
    "campaign", "initiated_by", "first_recall", "family_recalled"]].to_parquet(
    "../data/processed/test_preds.parquet", index=False)
print("Saved")

Saved


In [13]:
K = 100   # alerts reviewed per month

pos = te[te["target"] == 1].copy()
pos["month_end"] = pos["month"] + pd.offsets.MonthBegin(1)

recalls = pos.groupby("campaign").agg(
    initiated_by=("initiated_by", "first"),
    recall_date=("first_recall", "min"),
)
recalls["recall_date"] = pd.to_datetime(recalls["recall_date"])

def recall_level(rank_col, label):
    flagged = pos[pos[rank_col] <= K]
    first_flag = flagged.groupby("campaign")["month_end"].min()
    out = recalls.copy()
    out["flagged"] = out.index.isin(first_flag.index)
    out["lead_months"] = (out["recall_date"] - first_flag.reindex(out.index)).dt.days / 30.4
    summary = out.groupby("initiated_by").agg(
        recalls=("flagged", "size"),
        flagged=("flagged", "sum"),
        median_lead_months=("lead_months", "median"),
    )
    summary["pct_flagged"] = (100 * summary["flagged"] / summary["recalls"]).round(1)
    summary["model"] = label
    return summary.reset_index()

lead = pd.concat([
    recall_level("rank_struct", "Structural only"),
    recall_level("rank", "Full model"),
])
print(lead[["model", "initiated_by", "recalls", "flagged", "pct_flagged",
            "median_lead_months"]].round(1).to_string(index=False))
lead.to_csv("../results/lead_time_by_initiator.csv", index=False)

          model initiated_by  recalls  flagged  pct_flagged  median_lead_months
Structural only          MFR      507       44          8.7                 4.3
Structural only          ODI      132       10          7.6                 5.1
Structural only         OVSC        9        1         11.1                 5.1
     Full model          MFR      507       53         10.5                 4.4
     Full model          ODI      132       23         17.4                 4.1
     Full model         OVSC        9        0          0.0                 NaN


In [10]:
con.close()